# 04. Abliteration, Refusal Surgery, Circuit Breakers, and Sleeper Agents

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Arditi et al. (2024 *Refusal in Language Models Is Mediated by a Single Direction*); FailSpy (2024 *Abliteration: Weight Surgery for LLMs*); Turner et al. (2023 *Activation Addition / CAA*); Zou et al. (2024 *Improving Alignment and Robustness via Circuit Breakers*); Greenblatt et al. (2024 *Sleeper Agents: Training Deceptive LLMs That Persist Through Safety Training*).
- **Purpose:** Formulate and evaluate representation surgery and anti-jailbreak defenses: refusal direction extraction, weight-space orthogonal projection (abliteration), contrastive activation addition (CAA/RepE), Safety Circuit Breakers, and backdoor sleeper agent probing.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Machine Unlearning SISA Influence Scrub NPO and RMU](03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Safe RL Constrained MDP PPO Lagrangian CVaR and CBF](05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not assume 1D refusal alignment constitutes secure safety (open-weight models can be completely stripped of refusal in $<1\text{s}$ via single-matrix projection); do not abliterate weights without verifying zero degradation on benign reasoning manifolds.

## Mathematical Foundations & Governing Equations

1. **Refusal Direction Extraction**: $\hat{r} = \frac{\mu_{\text{harmful}} - \mu_{\text{harmless}}}{\|\mu_{\text{harmful}} - \mu_{\text{harmless}}\|_2}$
2. **Orthogonal Weight Surgery (Abliteration)**: $W' = W - \hat{r} (\hat{r}^\top W) \implies \hat{r}^\top W' = 0$
3. **Contrastive Activation Addition (CAA)**: $h' = h + \alpha \cdot \hat{v}$
4. **Safety Circuit Breakers**: $\mathcal{L}_{\text{CB}} = \mathbb{E}[\cos^2(h_h, h_{\text{target}})] + \lambda \cdot \mathbb{E}[\|h_b - h_{\text{ref}}\|_2^2]$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def extract_refusal(act_h, act_b):
    diff = torch.mean(act_h, dim=0) - torch.mean(act_b, dim=0)
    return diff / (torch.norm(diff, p=2) + 1e-8)

def abliterate(w, r):
    r_col = r.view(-1, 1)
    return w - r_col @ (r_col.t() @ w)

def caa_steer(h, steer_vec, alpha=2.0):
    return h + float(alpha) * steer_vec

def calc_cb_loss(h_h, h_t, h_b, h_ref_b, lam=1.5):
    dot = torch.sum(h_h * h_t, dim=-1)
    denom = (torch.norm(h_h, p=2, dim=-1) + 1e-8) * (torch.norm(h_t, p=2, dim=-1) + 1e-8)
    l_brk = torch.mean((dot / denom) ** 2)
    l_bng = torch.mean((h_b - h_ref_b) ** 2)
    return l_brk + float(lam) * l_bng

def probe_sleeper(h, w_probe, b_probe):
    return h @ w_probe + b_probe

In [ ]:
# Parity verification against golden reference tensors
r_hat = extract_refusal(weights["act_harmful"], weights["act_harmless"])
assert torch.max(torch.abs(r_hat - weights["refusal_dir_ref"])) < 1e-5

w_abl = abliterate(weights["w_mlp"], r_hat)
assert torch.max(torch.abs(w_abl - weights["w_ablated_ref"])) < 1e-5

h_str = caa_steer(weights["h_query"], r_hat, 2.0)
assert torch.max(torch.abs(h_str - weights["h_steered_ref"])) < 1e-5

cb_loss = calc_cb_loss(weights["h_harmful"], weights["h_target_ortho"], weights["h_benign"], weights["h_ref_benign"], 1.5)
assert torch.max(torch.abs(cb_loss - weights["cb_loss_ref"])) < 1e-5

p_scores = probe_sleeper(weights["h_query"], weights["probe_w"], weights["probe_b"])
assert torch.max(torch.abs(p_scores - weights["probe_scores_ref"])) < 1e-5
print("All abliteration, CAA, Circuit Breakers, and sleeper probe assertions passed!")

In [ ]:
# Benchmarking representation surgery operations
t0 = time.perf_counter()
for _ in range(500):
    r_h = extract_refusal(weights["act_harmful"], weights["act_harmless"])
    _ = abliterate(weights["w_mlp"], r_h)
    _ = caa_steer(weights["h_query"], r_h, 2.0)
    _ = calc_cb_loss(weights["h_harmful"], weights["h_target_ortho"], weights["h_benign"], weights["h_ref_benign"], 1.5)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused abliteration surgery + Circuit Breakers latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: The Fragility of 1D Refusal Alignment (Abliteration Immunity Contrast)
# An input query aligned exactly with the refusal direction r_hat:
x_harmful = r_hat.unsqueeze(0)  # Shape [1, D_in]

# 1. Under standard aligned model: strong activation in refusal direction
response_aligned = x_harmful @ weights["w_mlp"]
norm_aligned = torch.norm(response_aligned).item()

# 2. Under abliterated weights: the response in the refusal subspace is exactly 0.0!
response_ablated = x_harmful @ w_abl
norm_ablated = torch.norm(response_ablated).item()

print(f"Harmful Query Response Norm: Original Aligned = {norm_aligned:.4f} -> Abliterated = {norm_ablated:.4e}")
assert norm_aligned > 0.5, "Aligned model must strongly process refusal features!"
assert norm_ablated < 1e-6, "Abliteration must orthogonally eradicate refusal sensitivity in 1 step!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Refusal Abliteration** reveals that standard SFT/RLHF confines refusal into a fragile 1D direction stripped via $W' = W - \hat{r}\hat{r}^\top W$.
2. **Contrastive Activation Addition (CAA)** enables runtime steering of model demeanor and safety without permanent weight alterations.
3. **Safety Circuit Breakers** entangle safety into deep reasoning manifolds, preventing bypass via linear projection.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.md](../../markdown/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.md)
- **PyTorch 2.x Module:** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.py](../../pytorch2/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.py)
- **Functional JAX Module:** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.py](../../jax/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.py)
- **Native Rust Candle Track:** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.rs](../../candle/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.rs)
- **Hardware Kernel (CPU / WGSL):** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.rs](../../kernels/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.rs)
- **Golden SafeTensors:** [04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.safetensors](../../weights/11_ai_safety_privacy_and_verification/04_abliteration_refusal_surgery_circuit_breakers_and_sleeper_agents.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Machine Unlearning SISA Influence Scrub NPO and RMU](03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 05. Safe RL Constrained MDP PPO Lagrangian CVaR and CBF](05_safe_rl_constrained_mdp_ppo_lagrangian_cvar_and_cbf.ipynb) &rarr;